### PROJETO DE CONSULTA POR SIMILARIDADE  - SPDO ###

### Importando e padronizando os Dados ###

### Requirements ###

In [ ]:
import os, sys
# Garante que o cwd seja a raiz do projeto, independente de onde o notebook foi aberto
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

!pip install -q -r requirements.txt
%load_ext watermark

In [ ]:
import pandas as pd
import numpy as np
import warnings
from tqdm import tqdm
from src.data_process import padronizar_medida, preprocess_text, remove_stopwords, remover_palavras_duplicadas
warnings.filterwarnings("ignore")

### Carregamento dos Dados ###

In [ ]:
print("Carregando os dados...\n")
# Realiza a importação dos dados em csv e apresenta uma amostra com 10 registros aleatórios
df = pd.read_csv('data/input/consulta_bp.csv', engine='python')
df.sample(10)

### Tratamento da Base ###

In [ ]:
# Faz a verificação de valores nulos no Dataframe
df.isnull().sum()

In [ ]:
# Faz o preenchimento dos valores NaN com informações padronizadas de acordo com análise da base
df['DESCRICAO'] = df['DESCRICAO'].fillna('')
df['MARCA'] = df['MARCA'].fillna('SEM MARCA')
df['EMBALAGEM'] = df['EMBALAGEM'].fillna('')
df['QTD_MEDIDA'] = df['QTD_MEDIDA'].fillna('')

In [ ]:
# Faz a remoção de .0 a direita das colunas grp_insumo e qtd_medida, bem como a criação da coluna Unidade_Medida abv e com padrão completo
df['GRP_INSUMO'] = df['GRP_INSUMO'].astype(str).str.replace('.0', '')
df['MEDIDA_PAD'] = df['QTD_MEDIDA'].astype(str).str.replace('.0', '') + df['MEDIDA'].astype(str)
df['INSUMO_DESCRICAO'] = df['INSUMO'] + ' ' + df['DESCRICAO'].astype(str) + ' ' + '(' + df['EMBALAGEM'].astype(str) + ')'
df['MEDIDA_ABV'] = df['QTD_MEDIDA'].astype(str).str.replace('.0', '') + ' ' + df['CD_MEDIDA'].astype(str)
df.sample(10)

### Criando bases intermediarias ###

In [ ]:
# Gerar tabela de correlação de medidas
df_medida = df[['CD_MEDIDA','MEDIDA']]
df_medida = df_medida.drop_duplicates()
df_medida.to_csv('data/staging/medida_correlacao.csv', index=False)
df_medida.head()

In [ ]:
# Gerando um df padronizado que será utilizado para gerar os embeddings
df_embeddings = df[['GRP_INSUMO', 'CD_INSUMO', 'INSUMO_DESCRICAO', 'MARCA', 'MEDIDA_PAD']]
df_embeddings = remover_palavras_duplicadas(df_embeddings, ['INSUMO_DESCRICAO', 'MARCA', 'MEDIDA_PAD'])
df_embeddings = df_embeddings.rename(columns={'MEDIDA_PAD':'MEDIDA'})
df_embeddings.sample(10)

In [ ]:
# Gerando um df padronizado que será salvo e utilizado no final para entregar os resultados formatados
df_pad = df[['GRP_INSUMO', 'CD_INSUMO', 'INSUMO_DESCRICAO', 'MARCA', 'MEDIDA_ABV']]
df_pad = remover_palavras_duplicadas(df_pad, ['INSUMO_DESCRICAO', 'MARCA', 'MEDIDA_ABV'])
df_pad = df_pad.rename(columns={'MEDIDA_ABV':'MEDIDA'})
df_pad.to_csv('data/staging/df_pad.csv')
df_pad.sample(10)

### Padronização dos Dados ###

In [ ]:
# Padroniza as unidades de medida de volume, peso e comprimento para a mesma escala
df_embeddings["MEDIDA"] = df_embeddings["MEDIDA"].apply(padronizar_medida)
df_embeddings['MEDIDA'] = df_embeddings['MEDIDA'].astype(str).str.replace('.0', '')

In [ ]:
%%time
# Aplicar o pré-processamento de texto eliminando dados que são irrelevantes para comparação
print("Processando textos...")
tqdm.pandas()
for col in ['INSUMO_DESCRICAO', 'MARCA', 'MEDIDA']:
    df_embeddings[col] = df_embeddings[col].progress_apply(preprocess_text).progress_apply(remove_stopwords)

In [ ]:
df_embeddings.to_csv('data/staging/df_embeddings.csv')
df_embeddings.sample(10)

In [ ]:
%watermark --iversions